In [1]:
module chebyshev_method

    using LinearAlgebra

    export chebyshev_D
    # Chebyshev compute D = differentiation matrix, x = Chebyshev grid

    function  chebyshev_D(N)

        if N==0
            D = 0;
            x = 1;
            return D, x
        else
            θ = range(0,length=N+1,stop=pi)
            x = reshape(-cos.(θ), N+1, 1)
            c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
            X = repeat(x, 1, N+1);
            dX = X - X';
            D = (c * (1 ./ c)') ./ (dX .+ I(N+1));   # off-diagonal entries
            D = D - diagm(vec(sum(D, dims=2)));      # diagonal entries
            return D, x
        end
    end
end

Main.chebyshev_method

In [18]:
      using .chebyshev_method
      using LinearAlgebra
      using Plots
      N=299
      Re=10000.0    
      α=1   # Define Flow wave number and Reynolds number
      
      #Caculate the values and Vectors

      D,x=chebyshev_D(N)
      #O_S_method
      U_0=zeros(N+1,1)

      U_0[:,1]=ones(N+1,1)-x[:,1].*x[:,1]

      D2=D^2
      A=(D2-α^2*I(N+1))^2-(0+α*Re*im)*(D2.*U_0[:,1]-U_0[:,1].*(α^2*I(N+1))+2*I(N+1)) #O-S coefficient matrix
      B=(0+Re*α*im)*((α^2)*I(N+1)-D^2)
      # A[2,:]=D[1,:]
      # A[N,:]=D[N+1,:]
      NewA=A[3:N-1,3:N-1]
      NewB=B[3:N-1,3:N-1]
      # NewB[1,:].=0+0im
      # NewB[98,:].=0+0im
      values=eigvals!(NewA,NewB)
      B=eigen(NewA,NewB)
      Cr=zeros(N+1,1)
      Ci=zeros(N+1,1)
      Vectors=B.vectors
      Cr=real(values)#取合适范围的特征值
      Ci=imag(values)
      C=[Cr Ci]
      scatter(Cr,Ci,xlabel="Cr",ylabel="Ci",label="Values",ylims=[-1,0.2],leg=(:outertopright), titlefontsize=10)
      using DelimitedFiles
      writedlm("O-S_values.dat",C,'\t')